In [ ]:
import pandas as pd
import sienna
from evaluation import SchemaIntegrationEvaluation
import os
import numpy as np

### Aggregate results

In [ ]:
json_aggregated = {}
for metric in ["recall", "f1", "precision"]:
    # Read all recall csv files and concat all
    merged_metric_df = pd.DataFrame()
    for file_name in os.listdir("evaluation_summary"):
        if metric in file_name and ".csv" in file_name and "aggregated" not in file_name:
            df = pd.read_csv(f"evaluation_summary/{file_name}")
            merged_metric_df = pd.concat([merged_metric_df, df], ignore_index=True)
    
    # Group by parameters
    rows_average = []
    for group_name, group_df in merged_metric_df.groupby(["model_name", "prompt_name", "demonstration", "self_consistency", "generated_knowledge", "sequence_of_phases"]):
        for column in ['detect_tables_phase_entity', 'detect_tables_phase_attributes', 'schema_matching_phase_removed', 'grouping_phase', 'schema_integration_phase', 'final_integration_entity', 'final_integration_attributes', 'final_integration_mappings']:
            if '-' not in group_df[column].unique():
                group_df[column] = pd.to_numeric(group_df[column])
        if '-' in group_df['schema_matching_phase_removed'].unique():
            aggregations_mean = group_df.agg({'detect_tables_phase_entity': np.mean, 'detect_tables_phase_attributes': np.mean, 'grouping_phase': np.mean, 'schema_integration_phase': np.mean, 'final_integration_entity': np.mean, 'final_integration_attributes': np.mean, 'final_integration_mappings': np.mean})
        elif '-' in group_df['grouping_phase'].unique():
            aggregations_mean = group_df.agg({'detect_tables_phase_entity': np.mean, 'detect_tables_phase_attributes': np.mean, 'schema_matching_phase_removed': np.mean, 'schema_integration_phase': np.mean, 'final_integration_entity': np.mean, 'final_integration_attributes': np.mean, 'final_integration_mappings': np.mean})
        else:
            aggregations_mean = group_df.agg({'detect_tables_phase_entity': np.mean, 'detect_tables_phase_attributes': np.mean, 'schema_matching_phase_removed': np.mean, 'grouping_phase': np.mean, 'schema_integration_phase': np.mean, 'final_integration_entity': np.mean, 'final_integration_attributes': np.mean, 'final_integration_mappings': np.mean})

        # Create a row with the aggregations and the group parameters
        row = {**{f"{param}": value for param, value in zip(["model_name", "prompt_name", "demonstration", "self_consistency", "generated_knowledge", "sequence_of_phases"], group_name)}, **aggregations_mean.to_dict()}
        rows_average.append(row)
        

    metric_columns = ["detect_tables_phase_entity", "detect_tables_phase_attributes", "schema_matching_phase_removed", "grouping_phase", "schema_integration_phase", "final_integration_entity", "final_integration_attributes", "final_integration_mappings"]

    for row in rows_average:
        key = "_".join([str(row[param]) for param in ["model_name", "prompt_name", "self_consistency", "sequence_of_phases"]])
        if key not in json_aggregated:
            json_aggregated[key] = {}
            for metric_column in metric_columns:
                json_aggregated[key][metric_column] = {}
        
        for metric_column in metric_columns:
            if metric_column in row:
                json_aggregated[key][metric_column][metric] = row[metric_column]

sienna.save(json_aggregated, f"evaluation_summary/aggregated_all_cases_mean.json")

### Analysis

In [ ]:
# How many times were the loops used?
loops_per_use_case = {}
for file_name in os.listdir("logs/"):
    # if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" not in file_name:
    if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" not in file_name:
        log_file_name = "logs/" + file_name.replace("_evaluation.json", "").replace(".json", "")
        schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=log_file_name)
        if schema_evaluator.self_consistency:
            if schema_evaluator.run_info["sequence_of_phases"] == [ "detect_tables_phase", "schema_matching_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"] and schema_evaluator.self_consistency:
            # if schema_evaluator.run_info["sequence_of_phases"] == ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"] :
                schema_evaluator.run_evaluation()
                use_case = schema_evaluator.run_info["folder_name"]
                if use_case not in loops_per_use_case:
                    loops_per_use_case[use_case] = {"detect_tables_phase": 0, "grouping_phase": 0, "final_integration_phase": 0}
                for run in schema_evaluator.all_predictions:
                    if "detect_tables_phase" in schema_evaluator.all_predictions[run]:
                        loops_per_use_case[use_case]["detect_tables_phase"] += sum(schema_evaluator.all_predictions[run]["detect_tables_phase"]["missing_loop"].values())
                    if "grouping_phase" in schema_evaluator.all_predictions[run]:
                        loops_per_use_case[use_case]["grouping_phase"] += schema_evaluator.all_predictions[run]["grouping_phase"]["missed_tables_reassigned"]
                    if "final_integration_phase" in schema_evaluator.all_predictions[run] and "missed_attributes_loop" in schema_evaluator.all_predictions[run]["final_integration_phase"]:
                        loops_per_use_case[use_case]["final_integration_phase"] += schema_evaluator.all_predictions[run]["final_integration_phase"]["missed_attributes_loop"]

In [ ]:
# How is the schema matching loop change the integrated schema/mappings results?
schema_integration_res = {}
all_schema_evaluators = {}
for file_name in os.listdir("logs/"):
    # if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" not in file_name:
    if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" not in file_name:
        log_file_name = "logs/" + file_name.replace("_evaluation.json", "").replace(".json", "")
        schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=log_file_name)
        if schema_evaluator.self_consistency:
            if schema_evaluator.run_info["sequence_of_phases"] == [ "detect_tables_phase", "schema_matching_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"] and schema_evaluator.self_consistency:
            # if schema_evaluator.run_info["sequence_of_phases"] == ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"] :
                schema_evaluator.run_evaluation()
                eval_results = schema_evaluator.eval_results
                use_case = schema_evaluator.run_info["folder_name"]
                all_schema_evaluators[use_case] = schema_evaluator
                schema_integration_res[use_case] = eval_results["0"]["schema_integration_phase_extended"]


In [ ]:
# Integrated schemas analysis
rows = []
for use_case in schema_integration_res:
    removed = schema_integration_res[use_case]["integrated_schemas"]["schema_integration_phase_schema"]["eval_results_attributes"]["eval_stats"]
    if "integrated_attributes_no_removals" in schema_integration_res[use_case]:
        not_removed = schema_integration_res[use_case]["integrated_attributes_no_removals"]["schema_integration_phase_schema_no_removals"]["eval_results_attributes"]["eval_stats"]
    else:
        not_removed = removed
        # print(f"No separate evaluation for no removals in use case {use_case}, using the same results as with removals.")
    rows.append([use_case, removed["overall_precision"], removed["overall_recall"], removed["overall_f1"], not_removed["overall_precision"], not_removed["overall_recall"], not_removed["overall_f1"]])

In [ ]:
# Mappings analysis
rows = []
for use_case in schema_integration_res:
    removed = schema_integration_res[use_case]["integrated_schemas"]["schema_integration_phase_column_mappings"]
    if "integrated_attributes_no_removals" in schema_integration_res[use_case]:
        not_removed = schema_integration_res[use_case]["integrated_attributes_no_removals"]["schema_integration_phase_column_mappings"]
    else:
        not_removed = removed
        # print(f"No separate evaluation for no removals in use case {use_case}, using the same results as with removals.")
    rows.append([use_case, removed["overall_precision"], removed["overall_recall"], removed["overall_f1"], not_removed["overall_precision"], not_removed["overall_recall"], not_removed["overall_f1"]])

In [4]:
results_df = pd.DataFrame(rows, columns=["use_case", "precision_with_removals", "recall_with_removals", "f1_with_removals", "precision_no_removals", "recall_no_removals", "f1_no_removals"])

In [6]:
print("Average metrics with removals:")
print(f"Precision: {results_df['precision_with_removals'].mean()}")
print(f"Recall: {results_df['recall_with_removals'].mean()}")
print(f"F1: {results_df['f1_with_removals'].mean():.4f}")

Average metrics with removals:
Precision: 0.9334028832109027
Recall: 0.9334028832109027
F1: 0.9334


In [7]:
print("Average metrics without removals:")
print(f"Precision: {results_df['precision_no_removals'].mean()}")
print(f"Recall: {results_df['recall_no_removals'].mean()}")
print(f"F1: {results_df['f1_no_removals'].mean()}")

Average metrics without removals:
Precision: 0.8202896173481842
Recall: 0.8202896173481842
F1: 0.8202896173481842
